# DS4 Step 2, Path F' -- Joint Taxonomy (Path A' x Path C')

**Notebook:** `133_ds4_hierarchical_phenotype.ipynb`  
**Plan:** `outputs/reports/stage4/step2_phenotyping/ds4_step2_exploration_plan.md` SS4  
**Prerequisites:** Path A' (131, behavior-only, k=4) and Path C' (132, network-focused, k=3) -- both complete, real output loaded directly.

---

## Question

Path A' found DS4's Phase 1 taxonomy is mostly behavior-driven (ARI=0.664 vs Phase 1). Path C' found network structure is a nearly orthogonal, independently clean axis (ARI=0.047 vs Phase 1). If the two axes are genuinely independent, crossing them (4 behavior clusters x 3 network clusters = up to 12 cells) should reveal a richer joint structure than either alone -- matching the method Structured's own Path F used (crossing its Path A and Path C solutions), though DS4's real cross-tab shape and result are unknown until computed here.


## FLAG decisions

| Flag | Decision |
|---|---|
| FLAG 1 | Chi-square test of independence (not Fisher's exact -- table is 4x3, not 2x2; Structured's Path F used Fisher's exact specifically because its table was 2x2) |
| FLAG 2 | Report only cells with adequate count; flag (not hide) sparse cells |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import chi2_contingency

OUT_DIR = Path("outputs/model_outputs/ds4_phenotyping")
FIG_DIR = OUT_DIR / "figures"


In [ ]:
path_a = pd.read_csv(OUT_DIR / "path_a_feature_matrix.csv")[
    ["participant_id_canonical", "path_a_cluster"]]
path_c = pd.read_csv(OUT_DIR / "path_c_feature_matrix.csv")[
    ["participant_id_canonical", "path_c_cluster"]]

joint = path_a.merge(path_c, on="participant_id_canonical", how="inner")
print(f"Joint population: {len(joint)} (both Path A' and Path C' used the same "
      f"N=641 complete-case population, so this should be 641)")


In [ ]:
ct = pd.crosstab(joint["path_a_cluster"], joint["path_c_cluster"])
print("Joint cross-tab (rows=Path A' behavior cluster, cols=Path C' network cluster):")
print(ct)
print(f"\nMin cell count: {ct.values.min()}")


In [ ]:
chi2, p_value, dof, expected = chi2_contingency(ct)
print(f"Chi-square test of independence: chi2={chi2:.3f}, dof={dof}, p={p_value:.4f}")
print("\nExpected counts under independence:")
print(pd.DataFrame(expected, index=ct.index, columns=ct.columns).round(1))

min_expected = expected.min()
print(f"\nMin expected cell count: {min_expected:.1f} "
      f"({'OK, chi-square assumptions reasonably met' if min_expected >= 5 else 'CAUTION: some expected counts below 5, chi-square approximation may be unreliable'})")


In [ ]:
# Standardized (Pearson) residuals -- which cells drive any departure from independence
residuals = (ct.values - expected) / np.sqrt(expected)
resid_df = pd.DataFrame(residuals, index=ct.index, columns=ct.columns)
print("Standardized residuals (|value|>2 suggests that cell is over/under-represented):")
print(resid_df.round(2))


In [ ]:
joint.to_csv(OUT_DIR / "path_f_joint_assignments.csv", index=False)
ct.to_csv(OUT_DIR / "path_f_joint_crosstab.csv")

summary = pd.DataFrame([{
    "n": len(joint), "chi2": chi2, "dof": dof, "p_value": p_value,
    "min_observed_cell": int(ct.values.min()), "min_expected_cell": float(min_expected),
}])
summary.to_csv(OUT_DIR / "path_f_summary.csv", index=False)
print("Wrote path_f_joint_assignments.csv, path_f_joint_crosstab.csv, path_f_summary.csv")


In [ ]:
summary_lines = [
    "# DS4 Step 2, Path F' -- Model Run Summary", "",
    "**Notebook:** `133_ds4_hierarchical_phenotype.ipynb`  ",
    "**Plan:** `outputs/reports/stage4/step2_phenotyping/ds4_step2_exploration_plan.md`  ",
    "**Run date:** " + pd.Timestamp.now().strftime("%Y-%m-%d"), "",
    f"N={len(joint)}", "",
    "## Joint cross-tab", "", "```",
    ct.to_string(), "```", "",
    f"## Chi-square test: chi2={chi2:.3f}, dof={dof}, p={p_value:.4f}, "
    f"min_expected_cell={min_expected:.1f}", "",
    "## Standardized residuals", "", "```",
    resid_df.round(2).to_string(), "```",
]
with open(OUT_DIR / "path_f_model_run_summary.md", "w") as f:
    f.write("\n".join(summary_lines) + "\n")
print("Wrote path_f_model_run_summary.md")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(ct.values, cmap="Purples")
ax.set_xticks(range(ct.shape[1]))
ax.set_xticklabels([f"net {c}" for c in ct.columns])
ax.set_yticks(range(ct.shape[0]))
ax.set_yticklabels([f"behav {r}" for r in ct.index])
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        ax.text(j, i, ct.values[i, j], ha="center", va="center",
                color="white" if ct.values[i, j] > ct.values.max()/2 else "black")
ax.set_xlabel("Path C' network cluster")
ax.set_ylabel("Path A' behavior cluster")
ax.set_title(f"DS4 Path F': joint taxonomy (chi2 p={p_value:.4f})")
fig.colorbar(im, ax=ax, label="n participants")
fig.tight_layout()
fig.savefig(FIG_DIR / "path_f_fig1_joint_heatmap.png", dpi=150)
plt.close(fig)


## Interpretation and limits

**Read the cells above for live numbers.**

- A significant chi-square (p<0.05) with adequate expected cell counts means the behavior and network axes are NOT fully independent -- some combinations are over/under-represented beyond what independence would predict. A non-significant result means the two axes genuinely cross freely (any behavior type can co-occur with any network type at roughly the rate independence predicts).
- Do not over-interpret specific cells with low expected counts (flagged above) -- 12 cells across N=641 means some cells are inherently sparse.
- This closes the DS4 Step 2 further-exploration arc (Paths A'/C'/F'). Combined with Phase 1, DS4's phenotype space is now characterized as: a dominant behavioral/thermodynamic axis (Path A' recovers most of Phase 1), plus a real, largely independent network axis (Path C'), with their joint structure characterized here.


## Forward connections

DS4 Step 2 exploration (Phase 1 + Paths A'/C'/F') is now complete. Not yet started: DS4 Step 3 (personalization), or a DS4-vs-DS5 phenotype-structure synthesis.
